In [11]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
corpus = [
    "deep learning is a subset of machine learning",
    "recurrent neural networks are great for sequential data",
    "lstm networks solve the vanishing gradient problem",
    "natural language processing uses word embeddings",
    "deep neural networks require large amounts of data",
    "machine learning models learn patterns from data"
]

In [40]:
vectorizer = tf.keras.layers.TextVectorization(output_mode='int')

vectorizer.adapt(corpus)

vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)

In [41]:
print("vocab_size: ",vocab_size)
print("vocabulary: ",vocab)

vocab_size:  36
vocabulary:  ['', '[UNK]', np.str_('networks'), np.str_('learning'), np.str_('data'), np.str_('of'), np.str_('neural'), np.str_('machine'), np.str_('deep'), np.str_('word'), np.str_('vanishing'), np.str_('uses'), np.str_('the'), np.str_('subset'), np.str_('solve'), np.str_('sequential'), np.str_('require'), np.str_('recurrent'), np.str_('processing'), np.str_('problem'), np.str_('patterns'), np.str_('natural'), np.str_('models'), np.str_('lstm'), np.str_('learn'), np.str_('large'), np.str_('language'), np.str_('is'), np.str_('great'), np.str_('gradient'), np.str_('from'), np.str_('for'), np.str_('embeddings'), np.str_('are'), np.str_('amounts'), np.str_('a')]


In [42]:
# Convert sentences into integer sequences
input_sequences = []

for sentence in corpus:
    # Tokenize sentence (shape: [num_words])
    tokenized = vectorizer([sentence])[0].numpy()
    # Strip any trailing zeros
    tokenized = tokenized[tokenized > 0]
    
    # Generate sub-sequences
    for i in range(1, len(tokenized)):
        n_gram = tokenized[:i+1]
        input_sequences.append(n_gram)

# Determine maximum sequence length
max_seq_len = max(len(seq) for seq in input_sequences)

# Pad sequences to uniform length so they fit into a NumPy array
padded_sequences = tf.keras.utils.pad_sequences(
    input_sequences, 
    maxlen=max_seq_len, 
    padding='pre'  # Pad at the beginning so the target word is at the very end
)

# Inputs are all tokens except the last; Target is the last token
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (38, 7)
y shape: (38,)


In [43]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(max_seq_len - 1,)),
    # Use mask_zero=True to skip pre-padding
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=16, mask_zero=True),
    tf.keras.layers.LSTM(units=32),
    tf.keras.layers.Dense(units=vocab_size, activation='softmax')
])

In [33]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)              │ (None, 7, 16)               │             576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (None, 32)                  │           6,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 7)                   │             231 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,079 (27.65 KB)

 Trainable params: 7,079 (27.65 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)              │ (None, 7, 16)               │             576 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_5 (LSTM)                        │ (None, 32)                  │           6,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 36)                  │           1,188 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,036 (31.39 KB)

 Trainable params: 8,036 (31.39 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
model.fit(X, y, epochs=120, verbose=1)

Epoch 1/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.0000e+00 - loss: 3.5845
Epoch 2/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.1053 - loss: 3.5584
Epoch 3/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.0789 - loss: 3.5325
Epoch 4/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0789 - loss: 3.4913
Epoch 5/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.0789 - loss: 3.4083
Epoch 6/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.0789 - loss: 3.2913
Epoch 7/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.0789 - loss: 3.2045
Epoch 8/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0789 - loss: 3.2103
Epoch 9/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1316 - loss: 3.1445
Epoch 10/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.1842 - loss: 3.0385
Epoch 11/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.2105 - loss: 2.9571
Epoch 12/120
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3158 

In [46]:
print("Training complete! Final loss:", model.evaluate(X, y, verbose=0)[0])

Training complete! Final loss: 0.07498802244663239


In [47]:
def generate_text(seed_text, next_words=4):
    result = seed_text
    
    for _ in range(next_words):
        # 1. Vectorize seed text
        token_list = vectorizer([result])[0].numpy()
        token_list = token_list[token_list > 0]
        
        # 2. Pad to match model input shape
        token_list = tf.keras.utils.pad_sequences(
            [token_list], 
            maxlen=max_seq_len - 1, 
            padding='pre'
        )
        
        # 3. Predict next token probabilities
        probs = model.predict(token_list, verbose=0)[0]
        predicted_idx = np.argmax(probs)
        
        # 4. Map index back to word
        predicted_word = vocab[predicted_idx]
        if not predicted_word or predicted_word == '[UNK]':
            break
            
        result += " " + predicted_word
        
    return result

# Test generation
print(generate_text("deep learning", next_words=5))
print(generate_text("lstm networks", next_words=4))

deep learning is a subset of machine
lstm networks solve the vanishing gradient
